In [ ]:
# Comprehensive EDA for analysis_dataset.csv

from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
# ---------- Load data ----------
candidate_paths = [
    Path("analysis_dataset.csv"),
    Path("Data Analysis/analysis_dataset.csv"),
    Path("/Users/muneebahmed/Projects/ReSpace-AI/Data Analysis/analysis_dataset.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("analysis_dataset.csv not found in expected locations.")

df = pd.read_csv(csv_path, parse_dates=["processed_at"])
print(f"Loaded: {csv_path} | shape={df.shape}")
display(df.head(3))

In [ ]:
# ---------- Basic quality checks ----------
print("\n--- Basic Data Quality ---")
print(df.info())

missing = df.isna().sum().sort_values(ascending=False)
print("\nMissing values per column:")
display(missing.to_frame("missing_count"))

print("\nDuplicate checks:")
print("Duplicate id:", df["id"].duplicated().sum())
print("Duplicate original_id:", df["original_id"].duplicated().sum())
print("Duplicate image_url:", df["image_url"].duplicated().sum())
print("Duplicate local_path:", df["local_path"].duplicated().sum())

# Augmentation completeness per original sample
aug_per_original = df.groupby("original_id")["aug_type"].nunique().sort_values()
print("\nAugmentation types per original_id (value counts):")
display(aug_per_original.value_counts().sort_index().to_frame("num_original_ids"))

In [ ]:
# ---------- High-level distributions ----------
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sns.countplot(data=df, x="room_type", order=df["room_type"].value_counts().index, ax=axes[0, 0])
axes[0, 0].set_title("Room Type Distribution")
axes[0, 0].tick_params(axis="x", rotation=30)

sns.countplot(data=df, x="color_theme", order=df["color_theme"].value_counts().index, ax=axes[0, 1])
axes[0, 1].set_title("Color Theme Distribution")
axes[0, 1].tick_params(axis="x", rotation=30)

sns.countplot(data=df, x="use_case", order=df["use_case"].value_counts().index[:12], ax=axes[0, 2])
axes[0, 2].set_title("Top Use Cases")
axes[0, 2].tick_params(axis="x", rotation=45)

sns.countplot(data=df, x="lighting", order=df["lighting"].value_counts().index, ax=axes[1, 0])
axes[1, 0].set_title("Lighting Distribution")
axes[1, 0].tick_params(axis="x", rotation=30)

sns.countplot(data=df, x="color_palette", order=df["color_palette"].value_counts().index, ax=axes[1, 1])
axes[1, 1].set_title("Color Palette Distribution")
axes[1, 1].tick_params(axis="x", rotation=30)

sns.countplot(data=df, x="aug_type", order=df["aug_type"].value_counts().index, ax=axes[1, 2])
axes[1, 2].set_title("Augmentation Type Distribution")
axes[1, 2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# ---------- Cross-category analysis ----------
room_use = pd.crosstab(df["room_type"], df["use_case"])
plt.figure(figsize=(14, 6))
sns.heatmap(room_use, cmap="Blues", annot=False)
plt.title("Room Type vs Use Case (Counts)")
plt.ylabel("Room Type")
plt.xlabel("Use Case")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

light_palette = pd.crosstab(df["lighting"], df["color_palette"])
plt.figure(figsize=(8, 4))
sns.heatmap(light_palette, cmap="YlOrBr", annot=True, fmt="d")
plt.title("Lighting vs Color Palette")
plt.tight_layout()
plt.show()

# Normalized stacked composition: room_type by color_theme
room_color = pd.crosstab(df["room_type"], df["color_theme"], normalize="index")
room_color.plot(kind="bar", stacked=True, figsize=(12, 5), colormap="tab20")
plt.title("Color Theme Composition within Room Types")
plt.ylabel("Proportion")
plt.xlabel("Room Type")
plt.legend(title="Color Theme", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Time analysis ----------
df["processed_date"] = df["processed_at"].dt.date
df["processed_hour"] = df["processed_at"].dt.hour

daily = df.groupby("processed_date").size()
hourly = df.groupby("processed_hour").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
daily.plot(ax=axes[0], marker="o")
axes[0].set_title("Records Processed per Day")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Count")

hourly.plot(kind="bar", ax=axes[1], color="teal")
axes[1].set_title("Records Processed by Hour (UTC)")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# ---------- Furniture analysis ----------
# Split furniture list into normalized rows
furniture_long = (
    df.assign(furniture_item=df["furniture"].fillna("").str.split(","))
      .explode("furniture_item")
)
furniture_long["furniture_item"] = furniture_long["furniture_item"].str.strip()
furniture_long = furniture_long[furniture_long["furniture_item"] != ""]

top_furniture = furniture_long["furniture_item"].value_counts().head(12)
plt.figure(figsize=(10, 5))
sns.barplot(x=top_furniture.values, y=top_furniture.index, palette="viridis")
plt.title("Top Furniture Items")
plt.xlabel("Frequency")
plt.ylabel("Furniture Item")
plt.tight_layout()
plt.show()

# Furniture co-occurrence for top 8 items
top_items = set(furniture_long["furniture_item"].value_counts().head(8).index)
binary = (
    furniture_long[furniture_long["furniture_item"].isin(top_items)]
    .assign(v=1)
    .pivot_table(index="id", columns="furniture_item", values="v", aggfunc="max", fill_value=0)
)
co_occ = binary.T.dot(binary)
plt.figure(figsize=(8, 6))
sns.heatmap(co_occ, annot=True, fmt="d", cmap="mako")
plt.title("Furniture Co-occurrence Matrix (Top 8)")
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Useful summary tables ----------
print("\nTop (room_type, use_case, color_theme) combinations:")
top_combo = (
    df.groupby(["room_type", "use_case", "color_theme"])
      .size()
      .sort_values(ascending=False)
      .head(15)
      .rename("count")
      .reset_index()
)
display(top_combo)

print("\nStatus distribution:")
display(df["status"].value_counts().to_frame("count"))